### Structure output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic

Pydantic models provide the richest fearture set with field validation, descriptions, and nested structure.

In [29]:
import os
from langchain.chat_models import init_chat_model
from pydantic import BaseModel, Field

# Ensure your API key is set properly
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

# CHANGE HERE: Use 'model=' and include the 'groq:' prefix with the exact model ID
# CHANGE HERE: Use 3.1 instead of 3.2 for the 8B instant model
model = init_chat_model(model="groq:llama-3.1-8b-instant")

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    director: str = Field(description="The director of the movie")
    year: int = Field(description="The year the movie was released")
    genre: str = Field(description="The genre of the movie")
    rating: float = Field(description="The rating of the movie")

# Bind the Pydantic schema to the model
model_with_structuredm = model.with_structured_output(Movie)

# Invoke the model
response = model_with_structuredm.invoke("Provide the details of the movie Fight Club")
print(response)

title='Fight Club' director='David Fincher' year=1999 genre='Thriller' rating=8.8


### Message output alongside Parsed structure

In [32]:
from pydantic import BaseModel, Field
class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    director: str = Field(description="The director of the movie")
    year: int = Field(description="The year the movie was released")
    genre: str = Field(description="The genre of the movie")
    rating: float = Field(description="The rating of the movie")

model_with_structuredm = model.with_structured_output(Movie, include_raw=True)

response = model_with_structuredm.invoke("Provide the details of the movie Fight Club")
print(response)



{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'db3j0atmt', 'function': {'arguments': '{"director":"David Fincher","genre":"Action, Drama, Thriller","rating":8.8,"title":"Fight Club","year":1999}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 294, 'total_tokens': 339, 'completion_time': 0.08265959, 'completion_tokens_details': None, 'prompt_time': 0.020886805, 'prompt_tokens_details': None, 'queue_time': 0.047846525, 'total_time': 0.103546395}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019c7b17-1e09-7170-8cad-68d8f268f723-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'David Fincher', 'genre': 'Action, Drama, Thriller', 'rating': 8.8, 'title': 'Fight Club', 'year': 1999}, 'id': 'db3j0atmt', 'type': 'tool_call'}], invalid_tool_calls=[

### NESTED STRUCTURE

In [33]:
from pydantic import BaseModel, Field
class Actor(BaseModel):
    name: str 
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None,description="The budget of the movie in millions of dollars")

    

In [40]:

model_with_structuredm = model.with_structured_output(Movie)

response = model_with_structuredm.invoke("Provide the details of the movie Fight Club")
print(response)


title='Fight Club' director='David Fincher' year=1999 genre='Action, Drama' rating=8.8


### TypedDict

TypedDict provides a simpler alternative using python's built in typing, ideal when you don't need the runtime validation

In [44]:
from typing_extensions import TypedDict, Annotated


In [45]:
class MovieDict(TypedDict):
    """A movie with details"""
    title: Annotated[str,...,"The title of the movie"]
    year: Annotated[int,...,"The year of release"]
    rating: Annotated[float,...,"The rating of the movie"]
    director: Annotated[str,...,"The director of the movie"]
    

In [46]:

model_withtypedict = model.with_structured_output(Movie)

response = model_with_structuredm.invoke("Provide the details of the movie Fight Club")
print(response)


title='Fight Club' year=1999 cast=[Actor(name='Brad Pitt', role='Tyler Durden'), Actor(name='Edward Norton', role='The Narrator')] genres=['Drama', 'Thriller'] budget=63.0


### Structured Output
.
.
.
.